# social_text 全年推理结果分析

本 notebook 分析**全年 12 个月**的推理结果（`artifacts/social_text_probs_YYYYMM.csv`）。
推理本身由 `scripts/infer_social_text.py` 完成（双卡 fp16），本册只读产物做分析：
逐月概况 / 概率分布 / 日度-个股聚合 / pooling 对比 / 可视化。

**契约提醒（模型输出未确认项）：**
- 标签 `class_0/1` 语义未确认，**不要**命名为正面/负面/看多/看空；
- pooling 未确认（候选 `cls / pooler / masked_mean`），默认 `cls`，本册末尾有对比；
- 结果为 fp16，与 fp32 概率约有 ~1e-3 差异。

**前置**：全年 12 个结果 CSV 已生成（`artifacts/social_text_probs_202401..202412.csv`）。
若只生成了部分月份，cell 2 会加载当前已有的标准月份文件。

In [ ]:
# 环境与导入
%matplotlib inline
import os, re, sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path(os.environ.get("PROJECT_ROOT", Path.cwd())).resolve()
if not (ROOT / "configs" / "model.yaml").is_file():
    raise FileNotFoundError("请从仓库根目录启动 notebook，或设置 PROJECT_ROOT")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

DATA_DIR = ROOT / "filtered_data"
OUT_DIR = ROOT / "artifacts"
print("python:", sys.version.split()[0], "| numpy:", np.__version__, "| pandas:", pd.__version__)

In [ ]:
# 加载全年推理结果(12个月 concat, 排除 --limit 小样本产物)
pattern = re.compile(r"social_text_probs_(\d{6})\.csv$")
Months = sorted(m.group(1) for f in os.listdir(OUT_DIR) if (m := pattern.fullmatch(f)))
# 想快速试跑可只取前几个月: Months = Months[:2]
print("将加载月份:", Months)

frames = []
for m in Months:
    p = OUT_DIR / f"social_text_probs_{m}.csv"
    df = pd.read_csv(p, dtype={"class_0_prob": np.float32, "class_1_prob": np.float32})
    df["month"] = f"{m[:4]}-{m[4:]}"
    frames.append(df)
if not frames:
    raise RuntimeError("没有可用的结果 CSV, 请先运行 scripts/infer_social_text.py")

res = pd.concat(frames, ignore_index=True)
del frames
print("合并完成: shape =", res.shape, "| 月份数 =", len(Months), "| 内存 ~%.1f GB" % (res.memory_usage(deep=True).sum()/1e9))
print("列:", list(res.columns))
res.head(3)

In [ ]:
# 结果体检: 逐月行数 / NaN / 概率和 / 来源 / 日期对齐
print("逐月行数:")
print(res.groupby("month").size().rename("n_texts").to_string())
print()
print("NaN 数(class_0/1):", int(res[["class_0_prob", "class_1_prob"]].isna().sum().sum()))
s = res["class_0_prob"] + res["class_1_prob"]
print("概率和 max|sum-1|:", round(abs(s - 1).max(), 6))
print()
print("来源分布:")
print(res["source"].value_counts().to_string())
print()
print("各月 available_date 覆盖的交易日数:")
print(res.groupby("month")["available_date"].nunique().rename("n_trading_days").to_string())
print()
print("date -> available_date 抽样:")
print(res[["date", "available_date"]].drop_duplicates().sort_values("date").head(8).to_string())

In [ ]:
# class_1_prob 分布: 整体 + 逐月
print("class_1_prob 整体分布:")
print(res["class_1_prob"].describe().round(4).to_string())
print()
mm = res.groupby("month")["class_1_prob"].agg(["mean", "median", "std", "count"])
print("逐月 class_1_prob:")
print(mm.round(4).to_string())

In [ ]:
# 可视化: 直方图 / 逐月情绪均值 / 样本个股日度时序
fig, axes = plt.subplots(1, 3, figsize=(17, 4))

# 1) 直方图(抽样100万, 避免90M点过重)
samp = res["class_1_prob"].sample(min(1_000_000, len(res)), random_state=0)
axes[0].hist(samp, bins=60)
axes[0].set_title("class_1_prob 分布(抽样100万)")
axes[0].set_xlabel("class_1_prob")

# 2) 逐月情绪均值
mm = res.groupby("month")["class_1_prob"].mean()
axes[1].plot(mm.index, mm.values, marker="o")
axes[1].set_title("逐月 class_1_prob 均值")
axes[1].tick_params(axis="x", rotation=45, labelsize=8)

# 3) 样本个股日度情绪均值时序
top_sym = res["symbol"].value_counts().index[0]
d = (res[res["symbol"] == top_sym]
     .groupby("available_date")["class_1_prob"].mean().sort_index())
axes[2].plot(d.index.astype(str), d.values, lw=0.8)
axes[2].set_title(f"{top_sym} 日度 class_1_prob 均值")
axes[2].tick_params(axis="x", rotation=90, labelsize=6)
plt.tight_layout()
plt.show()

In [ ]:
# 日度-个股情绪分数(全年聚合) + 落盘
daily = (res.groupby(["available_date", "symbol"])["class_1_prob"]
         .agg(["mean", "median", "std", "count"]).reset_index())
daily.columns = ["available_date", "symbol", "sentiment_mean", "sentiment_median", "sentiment_std", "n_texts"]
daily = daily.sort_values(["available_date", "symbol"])
DAILY_PATH = OUT_DIR / "social_text_daily_2024.csv"
daily.to_csv(DAILY_PATH, index=False)
print(f"全年聚合: {daily.shape[0]} 行 (交易日 x 个股) -> {DAILY_PATH}")
daily.head()

---

## 附加：源文件体检（空文本 / 截断，可选）

对**原始源文件**（`filtered_data`，含 text）做抽样检查，与模型无关。按需运行。

In [ ]:
# 空文本 / 截断检查(抽样源文件)
SMONTH = "202403"
src = pd.read_csv(DATA_DIR / f"social_text_{SMONTH}.csv.gz",
                  usecols=["text"], nrows=2_000_000)
n_empty = int((src["text"].fillna("").str.strip() == "").sum())
print(f"{SMONTH} 抽样 200万: 空文本 {n_empty} ({n_empty/len(src):.2%})")

from transformers import BertTokenizerFast
from src.config import load_yaml_config
cfg = load_yaml_config(ROOT / "configs" / "model.yaml")
base = ROOT / cfg["paths"]["base_model_dir"]
tok = BertTokenizerFast.from_pretrained(base, local_files_only=True)
sub = src["text"].fillna("").head(2000).tolist()
lens = np.array([len(x) for x in tok(sub, add_special_tokens=True)["input_ids"]])
print(f"截断: 抽样 2000 条, >128 token 占比 {(lens > 128).mean():.2%}, 中位长度 {np.median(lens):.0f}")

---

## pooling 三候选对比（pooling 未确认 → 看输出一致性）

复用脚本的**预缓存**对同一样本分别跑 `cls / pooler / masked_mean` 前向，比较 `class_1_prob` 相关系数。
若缓存不存在会先自动跑一个 `--limit 小样本`（不影响正式结果）。

In [ ]:
# pooling 对比（纯前向，走预缓存，不 tokenize）
import subprocess, json
import torch
from src.config import load_yaml_config
from src.models.modeling import build_candidate

PMONTH = "202403"          # 抽样月份
SAMPLE = 3000
cfg = load_yaml_config(ROOT / "configs" / "model.yaml")
base = ROOT / cfg["paths"]["base_model_dir"]
ckpt = ROOT / cfg["paths"]["checkpoint"]

sample_cache = OUT_DIR / "token_cache" / f"social_text_{PMONTH}_lim{SAMPLE}_len128_ids.npy"
meta_path = Path(str(sample_cache) + ".meta.json")
if not (sample_cache.exists() and meta_path.exists()):
    print(f"生成样本缓存(--limit {SAMPLE})...")
    requested_device = "0" if torch.cuda.is_available() else "cpu"
    requested_dtype = "fp16" if torch.cuda.is_available() else "fp32"
    subprocess.run([sys.executable, str(ROOT / "scripts" / "infer_social_text.py"),
                    "--month", PMONTH, "--gpus", requested_device, "--limit", str(SAMPLE), "--dtype", requested_dtype],
                   cwd=ROOT, check=True)

with meta_path.open(encoding="utf-8") as f:
    meta = json.load(f)
pad = int(meta["pad_token_id"])
mm = np.load(sample_cache, mmap_mode="r")
ids = torch.from_numpy(mm[:SAMPLE].astype("int64"))
mask = (ids != pad)
device = "cuda" if torch.cuda.is_available() else "cpu"
ids = ids.to(device); mask = mask.to(device)

comp = {}
for p in ["cls", "pooler", "masked_mean"]:
    m = build_candidate(base, ckpt, pooling=p, pooling_confirmed=False,
                        device=device, dtype=(torch.float16 if device == "cuda" else torch.float32)).eval()
    with torch.inference_mode():
        probs = m(ids, attention_mask=mask).probabilities.float()[:, 1].cpu()
    comp[p] = probs.numpy()
    del m

cmp = pd.DataFrame(comp)
print(f"三种 pooling 的 class_1_prob（样本 {SAMPLE}）相关系数矩阵:")
print(cmp.corr().round(4).to_string())
print("\n均值:")
print(cmp.mean().round(4).to_string())

---

## 结论与下一步

- 全年 12 个月推理结果已在本册完成：**逐月概况 / 概率分布 / 日度-个股聚合(`artifacts/social_text_daily_2024.csv`) / pooling 对比 / 可视化**。
- 已知限制：标签语义未确认、pooling 未确认（见上对比）、fp16 概率与 fp32 有 ~1e-3 差异。
- 后续使用日度结果时，应继续以 `available_date` 为市场日期，并保留 `class_0/1` 中性命名，直到标签语义得到证据确认。